In [55]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


# Load your dataset
data = pd.read_csv(r"D:\Interview\NLP_NLTK\twitter_training.csv")

# Inspect
print(data.head())


   Tweet ID       entity sentiment  \
0      2401  Borderlands  Positive   
1      2401  Borderlands  Positive   
2      2401  Borderlands  Positive   
3      2401  Borderlands  Positive   
4      2401  Borderlands  Positive   

                                       Tweet content  
0  im getting on borderlands and i will murder yo...  
1  I am coming to the borders and I will kill you...  
2  im getting on borderlands and i will kill you ...  
3  im coming on borderlands and i will murder you...  
4  im getting on borderlands 2 and i will murder ...  


In [56]:
import nltk

# Download stopwords
nltk.download('stopwords')

# Download WordNet for lemmatization
nltk.download('wordnet')

# (Optional) Download 'omw-1.4' for better lemmatization coverage
nltk.download('omw-1.4')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mohan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mohan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mohan\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [57]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download resources once
# nltk.download('stopwords')
# nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_tweet(text):
    # Remove mentions, hashtags, URLs, non-letters
    text = re.sub(r"@\w+", "", str(text))
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)
    text = text.lower().strip()
    
    # Tokenize
    words = text.split()
    
    # Remove stopwords and lemmatize
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    
    return " ".join(words)

# Apply preprocessing
data['clean_tweet'] = data['Tweet content'].apply(clean_tweet)


In [58]:
#Encode Labels
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data['sentiment_label'] = le.fit_transform(data['sentiment'].astype(str)) # type: ignore
print(dict(zip(le.classes_, le.transform(le.classes_)))) # type: ignore

{'Irrelevant': np.int64(0), 'Negative': np.int64(1), 'Neutral': np.int64(2), 'Positive': np.int64(3)}


In [59]:
#Tokenizer
from tensorflow.keras.preprocessing.text import Tokenizer # type: ignore
from tensorflow.keras.preprocessing.sequence import pad_sequences # type: ignore



tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(data['clean_tweet'])

X = tokenizer.texts_to_sequences(data['clean_tweet'])
X = pad_sequences(X, maxlen=20)

y = data['sentiment_label']

In [60]:
# Save tokenizer and label encoder for future use (e.g., inference) 
import pickle
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)


In [61]:
# Train/Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [62]:
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense # type: ignore
from tensorflow.keras.layers import Bidirectional # type: ignore


vocab_size = 5000
max_len = 60

def build_rnn():
    model = Sequential()
    model.add(Embedding(vocab_size, 128, input_length=max_len))
    model.add(SimpleRNN(128))
    model.add(Dense(4, activation='softmax'))
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

def build_lstm():
    model = Sequential()
    model.add(Embedding(vocab_size, 128, input_length=max_len))
    model.add(LSTM(128))
    model.add(Dense(4, activation='softmax'))
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

def build_gru():
    model = Sequential()
    model.add(Embedding(vocab_size, 128, input_length=max_len))
    model.add(GRU(128))
    model.add(Dense(4, activation='softmax'))
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model




In [63]:
print("Training RNN...")
rnn_model = build_rnn()
rnn_model.fit(X_train, y_train, epochs=10, batch_size=64, validation_data=(X_test, y_test))
rnn_model.save("rnn_twitter_sentiment.keras")

print("Training LSTM...")
lstm_model = build_lstm()
lstm_model.fit(X_train, y_train, epochs=10, batch_size=64, validation_data=(X_test, y_test))
lstm_model.save("lstm_twitter_sentiment.keras")

print("Training GRU...")
gru_model = build_gru()
gru_model.fit(X_train, y_train, epochs=10, batch_size=64, validation_data=(X_test, y_test))
gru_model.save("gru_twitter_sentiment.keras")


Training RNN...


c:\Users\mohan\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 22s 20ms/step - accuracy: 0.6034 - loss: 0.9595 - val_accuracy: 0.7229 - val_loss: 0.7188
Epoch 2/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 16s 17ms/step - accuracy: 0.8032 - loss: 0.5196 - val_accuracy: 0.7400 - val_loss: 0.6709
Epoch 3/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 18s 19ms/step - accuracy: 0.8631 - loss: 0.3593 - val_accuracy: 0.7699 - val_loss: 0.6635
Epoch 4/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 17s 18ms/step - accuracy: 0.8872 - loss: 0.2894 - val_accuracy: 0.7557 - val_loss: 0.7583
Epoch 5/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 18s 19ms/step - accuracy: 0.9017 - loss: 0.2500 - val_accuracy: 0.7664 - val_loss: 0.7662
Epoch 6/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 19s 21ms/step - accuracy: 0.9092 - loss: 0.2311 - val_accuracy: 0.7603 - val_loss: 0.8278
Epoch 7/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 19s 21ms/step - accuracy: 0.9164 - loss: 0.2129 - val_accuracy: 0.7599 - val_loss: 0.8603
Epoch 8/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 20s 21ms/step - accuracy: 0.9173 - loss: 0.2060 - 

In [64]:
from sklearn.metrics import classification_report, accuracy_score
# Evaluate RNN
# Predictions
y_pred_rnn = rnn_model.predict(X_test)
y_pred_rnn = y_pred_rnn.argmax(axis=1)

# Metrics
print("RNN Accuracy:", accuracy_score(y_test, y_pred_rnn))
print("RNN Classification Report:\n")
print(classification_report(y_test, y_pred_rnn))

467/467 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
RNN Accuracy: 0.7611970275155654
RNN Classification Report:

              precision    recall  f1-score   support

           0       0.77      0.64      0.70      2592
           1       0.73      0.86      0.79      4519
           2       0.78      0.72      0.75      3596
           3       0.78      0.76      0.77      4230

    accuracy                           0.76     14937
   macro avg       0.76      0.75      0.75     14937
weighted avg       0.76      0.76      0.76     14937



In [65]:
# Evaluate LSTM

y_pred_lstm = lstm_model.predict(X_test)
y_pred_lstm = y_pred_lstm.argmax(axis=1)

print("LSTM Accuracy:", accuracy_score(y_test, y_pred_lstm))
print("LSTM Classification Report:\n")
print(classification_report(y_test, y_pred_lstm))

467/467 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step
LSTM Accuracy: 0.823056838722635
LSTM Classification Report:

              precision    recall  f1-score   support

           0       0.83      0.74      0.78      2592
           1       0.87      0.84      0.85      4519
           2       0.81      0.82      0.82      3596
           3       0.78      0.86      0.82      4230

    accuracy                           0.82     14937
   macro avg       0.82      0.81      0.82     14937
weighted avg       0.82      0.82      0.82     14937



In [66]:
# Evaluate GRU

y_pred_gru = gru_model.predict(X_test)
y_pred_gru = y_pred_gru.argmax(axis=1)

print("GRU Accuracy:", accuracy_score(y_test, y_pred_gru))
print("GRU Classification Report:\n")
print(classification_report(y_test, y_pred_gru))

467/467 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step
GRU Accuracy: 0.8170984802838589
GRU Classification Report:

              precision    recall  f1-score   support

           0       0.79      0.76      0.77      2592
           1       0.79      0.88      0.83      4519
           2       0.83      0.80      0.81      3596
           3       0.86      0.81      0.83      4230

    accuracy                           0.82     14937
   macro avg       0.82      0.81      0.81     14937
weighted avg       0.82      0.82      0.82     14937



In [67]:
# Tunning RNN with Bidirectional and Dropout
# from tensorflow.keras.layers import Dropout
from tensorflow.keras.optimizers import Adam # type: ignore
from tensorflow.keras.layers import Dropout # type: ignore

def build_rnn_tuned():
    model = Sequential()
    
    model.add(Embedding(vocab_size, 128, input_length=max_len))
    
    model.add(SimpleRNN(128, return_sequences=True))   # ✅ allow stacking
    model.add(SimpleRNN(64))
    
    model.add(Dropout(0.3))  # ✅ prevent overfitting
    
    model.add(Dense(4, activation='softmax'))
    
    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=Adam(learning_rate=0.001),
        metrics=['accuracy']
    )
    return model
print("Training Tuned RNN...")
rnn_model_Tuned = build_rnn_tuned()
rnn_model_Tuned.fit(X_train, y_train, epochs=10, batch_size=64, validation_data=(X_test, y_test))
rnn_model_Tuned.save("rnn_twitter_sentiment_tuned.keras")



Training Tuned RNN...
Epoch 1/10


c:\Users\mohan\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


934/934 ━━━━━━━━━━━━━━━━━━━━ 29s 28ms/step - accuracy: 0.5820 - loss: 1.0014 - val_accuracy: 0.7019 - val_loss: 0.7675
Epoch 2/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 25s 27ms/step - accuracy: 0.7841 - loss: 0.5691 - val_accuracy: 0.7547 - val_loss: 0.6474
Epoch 3/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 21s 23ms/step - accuracy: 0.8475 - loss: 0.4051 - val_accuracy: 0.7631 - val_loss: 0.6636
Epoch 4/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 23s 25ms/step - accuracy: 0.8759 - loss: 0.3247 - val_accuracy: 0.7670 - val_loss: 0.6977
Epoch 5/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 17s 18ms/step - accuracy: 0.8904 - loss: 0.2845 - val_accuracy: 0.7585 - val_loss: 0.7316
Epoch 6/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - accuracy: 0.9000 - loss: 0.2597 - val_accuracy: 0.7700 - val_loss: 0.7331
Epoch 7/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - accuracy: 0.9051 - loss: 0.2448 - val_accuracy: 0.7596 - val_loss: 0.7744
Epoch 8/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - accuracy: 0.9091 - loss: 0.2325 - val_accurac

In [68]:
# Tunning LSTM with Bidirectional and Dropout
def build_lstm_tuned():
    model = Sequential()
    
    model.add(Embedding(vocab_size, 128, input_length=max_len))
    
    model.add(Bidirectional(LSTM(128, return_sequences=True)))  # 🔥 upgrade
    model.add(LSTM(64))
    
    model.add(Dropout(0.3))
    
    model.add(Dense(4, activation='softmax'))
    
    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=Adam(learning_rate=0.001),
        metrics=['accuracy']
    )
    
    return model
print("Training Tuned LSTM...")
lstm_model_Tuned = build_lstm_tuned()
lstm_model_Tuned.fit(X_train, y_train, epochs=10, batch_size=64, validation_data=(X_test, y_test))
lstm_model_Tuned.save("lstm_twitter_sentiment_tuned.keras")


Training Tuned LSTM...
Epoch 1/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 71s 71ms/step - accuracy: 0.6079 - loss: 0.9668 - val_accuracy: 0.6928 - val_loss: 0.7823
Epoch 2/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 63s 68ms/step - accuracy: 0.7536 - loss: 0.6492 - val_accuracy: 0.7437 - val_loss: 0.6658
Epoch 3/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 62s 67ms/step - accuracy: 0.8072 - loss: 0.5078 - val_accuracy: 0.7726 - val_loss: 0.6072
Epoch 4/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 46s 49ms/step - accuracy: 0.8396 - loss: 0.4175 - val_accuracy: 0.7866 - val_loss: 0.5946
Epoch 5/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 69s 74ms/step - accuracy: 0.8650 - loss: 0.3482 - val_accuracy: 0.7903 - val_loss: 0.6229
Epoch 6/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 71s 76ms/step - accuracy: 0.8828 - loss: 0.2992 - val_accuracy: 0.8028 - val_loss: 0.6049
Epoch 7/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 71s 76ms/step - accuracy: 0.8966 - loss: 0.2646 - val_accuracy: 0.8123 - val_loss: 0.6058
Epoch 8/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 64s 69ms/step - accuracy: 0

In [69]:
# Tunning GRU with Bidirectional and Dropout
def build_gru_tuned():
    model = Sequential()
    
    model.add(Embedding(vocab_size, 128, input_length=max_len))
    
    model.add(Bidirectional(GRU(128, return_sequences=True)))  # 🔥 big boost
    model.add(GRU(64))
    
    model.add(Dropout(0.3))
    
    model.add(Dense(4, activation='softmax'))
    
    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=Adam(learning_rate=0.001),
        metrics=['accuracy']
    )
    
    return model
print("Training Tuned GRU...")
gru_model_Tuned = build_gru_tuned()
gru_model_Tuned.fit(X_train, y_train, epochs=10, batch_size=64, validation_data=(X_test, y_test))
gru_model_Tuned.save("gru_twitter_sentiment_tuned.keras")

Training Tuned GRU...
Epoch 1/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 56s 52ms/step - accuracy: 0.6164 - loss: 0.9402 - val_accuracy: 0.7092 - val_loss: 0.7440
Epoch 2/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 46s 49ms/step - accuracy: 0.7657 - loss: 0.6125 - val_accuracy: 0.7517 - val_loss: 0.6420
Epoch 3/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 58s 62ms/step - accuracy: 0.8180 - loss: 0.4777 - val_accuracy: 0.7799 - val_loss: 0.5952
Epoch 4/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 71s 50ms/step - accuracy: 0.8523 - loss: 0.3891 - val_accuracy: 0.7946 - val_loss: 0.5793
Epoch 5/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 46s 49ms/step - accuracy: 0.8752 - loss: 0.3277 - val_accuracy: 0.8049 - val_loss: 0.5761
Epoch 6/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 86s 53ms/step - accuracy: 0.8918 - loss: 0.2810 - val_accuracy: 0.8079 - val_loss: 0.5936
Epoch 7/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 45s 48ms/step - accuracy: 0.9043 - loss: 0.2469 - val_accuracy: 0.8108 - val_loss: 0.6255
Epoch 8/10
934/934 ━━━━━━━━━━━━━━━━━━━━ 47s 51ms/step - accuracy: 0.

In [70]:
from sklearn.metrics import classification_report, accuracy_score

def evaluate_model(model, X_test, y_test, name):
    # Predict probabilities
    y_pred = model.predict(X_test)
    
    # Convert to class index
    y_pred = y_pred.argmax(axis=1)
    
    # Accuracy
    acc = accuracy_score(y_test, y_pred)
    
    print(f"\n{name} Accuracy: {acc:.4f}")
    
    # Full report
    print(f"{name} Classification Report:\n")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

In [71]:
evaluate_model(rnn_model_Tuned, X_test, y_test, "RNN")
evaluate_model(lstm_model_Tuned, X_test, y_test, "LSTM")
evaluate_model(gru_model_Tuned, X_test, y_test, "GRU")

467/467 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step

RNN Accuracy: 0.7596
RNN Classification Report:

              precision    recall  f1-score   support

  Irrelevant       0.73      0.71      0.72      2592
    Negative       0.74      0.84      0.79      4519
     Neutral       0.75      0.74      0.75      3596
    Positive       0.81      0.72      0.76      4230

    accuracy                           0.76     14937
   macro avg       0.76      0.75      0.75     14937
weighted avg       0.76      0.76      0.76     14937

467/467 ━━━━━━━━━━━━━━━━━━━━ 9s 18ms/step

LSTM Accuracy: 0.8223
LSTM Classification Report:

              precision    recall  f1-score   support

  Irrelevant       0.83      0.75      0.79      2592
    Negative       0.84      0.87      0.86      4519
     Neutral       0.82      0.80      0.81      3596
    Positive       0.80      0.84      0.82      4230

    accuracy                           0.82     14937
   macro avg       0.82      0.81      0.82     14937
